In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeRegressor # used for the DT Stump
from sklearn.ensemble import AdaBoostRegressor
from sklearn.pipeline import Pipeline

---
## <u>Generate Dataset</u>

In [2]:
X, y = make_regression(   
    n_samples = 10000,    
    n_features = 15,     
    noise = 15,           
    n_informative = 12,  
    random_state = 42
)

In [3]:
X = pd.DataFrame(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [4]:
# 1. We create a base model (DT Stump)

dt_stump = DecisionTreeRegressor(
    max_depth = 1,                       # forces the dt stump to make the simplest of predictions 
    random_state = 42                    # for repeatability
)

# 2. Ada_Boost_Classifier

ada_regressor = AdaBoostRegressor(
    estimator = dt_stump,                # if estimator = 'None', AdaBoostClassifier makes dt_stump on its own exactly like this
    n_estimators = 100,                  # No of weak learners
    random_state = 42
) 

ada_regressor.fit(X_train, y_train)

# 3. Predict

y_train_pred = ada_regressor.predict(X_train)
y_test_pred = ada_regressor.predict(X_test)

In [5]:
print("For AdaBoost Regressor :-\n")

print("\nTraining scores :-")
print("Train R2 Score : ", r2_score(y_train, y_train_pred))

print("\nTesting scores :-")
print("Test R2 Score : ", r2_score(y_test, y_test_pred))

For AdaBoost Regressor :-


Training scores :-
Train R2 Score :  0.5119062376212891

Testing scores :-
Test R2 Score :  0.5096827704729685


In [8]:
# 1. make the pipeline

steps = [("ada", AdaBoostRegressor(
    random_state = 42,
    estimator = DecisionTreeRegressor(random_state = 42)
))]
pipeline = Pipeline(steps)

# 2. make parameter grid 

param_grid = {
    
    # paramters for the ada boost classifier
    "ada__n_estimators": [50, 100, 150],
    "ada__learning_rate": [0.1, 0.5, 1.0],

    # paramters for our estimator (DT stump) 
    "ada__estimator__max_depth": [2, 4, 6],             # as its a regression problem, just 1 level in DT stump would be very vague
    "ada__estimator__min_samples_split": [5, 10, 20],
    "ada__estimator__ccp_alpha": [0.1, 0.01, 0.001]
}

# 3. cross validation

ada_regressor_cv = RandomizedSearchCV(
    pipeline,
    param_grid,
    cv = 5,
    n_jobs = -1
    
)

# 4. train and predict

ada_regressor_cv.fit(X_train, y_train)
y_train_pred = ada_regressor_cv.predict(X_train)
y_test_pred = ada_regressor_cv.predict(X_test)

# 4. evaluate

print("For AdaBoost Regressor (hyperparameter tuning):-\n")
print("Best Parameters : ", ada_regressor_cv.best_params_)


print("\nTraining scores :-")
print("Train R2 Score : ", r2_score(y_train, y_train_pred))

print("\nTesting scores :-")
print("Test R2 Score : ", r2_score(y_test, y_test_pred))

For AdaBoost Regressor (hyperparameter tuning):-

Best Parameters :  {'ada__n_estimators': 150, 'ada__learning_rate': 0.5, 'ada__estimator__min_samples_split': 10, 'ada__estimator__max_depth': 6, 'ada__estimator__ccp_alpha': 0.1}

Training scores :-
Train R2 Score :  0.9113511030325063

Testing scores :-
Test R2 Score :  0.8604964331922699
